# Assignment 2: The Dirty Ledger — Exploratory Data Audit

**Dataset:** `teaching_prices_messy.csv`  
**Method:** Preserve the raw records and run diagnostics on a separate working copy.

This notebook is an audit, not a final cleaned dataset. It checks structure, missingness, parsing, duplicate records, repeated date–item–city keys, label consistency, observation counts, units, source labels, non-positive prices, and potential IQR outliers.

Price conversion is performed in a separate column. A comma is removed only when it is used in a correctly grouped thousands separator (for example, `1,250`). Malformed comma placement is not silently accepted. Original source columns are checked again at the end.

**Treatment rule:** flag and inspect first. Do not automatically delete rows, impute prices, recode item labels, or overwrite the CSV.

## Run the audit

Keep `teaching_prices_messy.csv` in the same folder as this notebook, then run the code cell below.

In [1]:
import pandas as pd

# 1. Load the source file and keep an untouched raw dataframe.
data_file = "teaching_prices_messy.csv"
df_raw = pd.read_csv(data_file)
df_work = df_raw.copy(deep=True)

required_columns = ["date", "item", "city", "price", "unit", "data_label"]
missing_columns = [column for column in required_columns if column not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("Target file:", data_file)
print("Dataset shape:", df_raw.shape)
print("Column headers:", df_raw.columns.tolist())

display(df_raw.head())
display(df_raw.dtypes.to_frame("source_dtype"))
display(df_raw.describe(include="all").T)

# 2. Parse dates and prices into additional working columns.
# The source columns are not overwritten.
df_work["date_parsed"] = pd.to_datetime(df_raw["date"], errors="coerce")

price_text = df_raw["price"].astype("string").str.strip()

# Remove commas only when they follow valid thousands grouping, e.g. 1,250
# or 1,250,000. Malformed comma placement is left unparsed and counted below.
valid_grouped_number = price_text.str.fullmatch(
    r"[+-]?\d{1,3}(?:,\d{3})+(?:\.\d+)?",
    na=False,
)
has_comma = price_text.str.contains(",", regex=False, na=False)
valid_price_format = (~has_comma) | valid_grouped_number

price_text_for_conversion = price_text.where(valid_price_format)
price_text_for_conversion = price_text_for_conversion.str.replace(
    ",", "", regex=False
)
df_work["price_numeric"] = pd.to_numeric(
    price_text_for_conversion, errors="coerce"
)

original_missing = int(df_raw.isna().sum().sum())
price_conversion_errors = int(
    (df_raw["price"].notna() & df_work["price_numeric"].isna()).sum()
)
date_conversion_errors = int(
    (df_raw["date"].notna() & df_work["date_parsed"].isna()).sum()
)

# 3. Overall diagnostic metrics.
raw_key_columns = ["date", "item", "city"]
complete_raw_keys = df_raw[raw_key_columns].notna().all(axis=1)
raw_key_duplicate_mask = pd.Series(False, index=df_raw.index)
raw_key_duplicate_mask.loc[complete_raw_keys] = df_raw.loc[
    complete_raw_keys, raw_key_columns
].duplicated(keep=False)

diagnostic_metrics = {
    "total_rows": len(df_raw),
    "total_columns": len(df_raw.columns),
    "original_missing_cells": original_missing,
    "exact_duplicate_occurrences": int(df_raw.duplicated().sum()),
    "raw_composite_key_duplicate_records": int(raw_key_duplicate_mask.sum()),
    "price_conversion_errors": price_conversion_errors,
    "non_positive_numeric_prices": int(
        (df_work["price_numeric"].dropna() <= 0).sum()
    ),
    "date_conversion_errors": date_conversion_errors,
    "unique_dates": int(df_raw["date"].nunique(dropna=True)),
    "unique_cities": int(df_raw["city"].nunique(dropna=True)),
}
display(pd.Series(diagnostic_metrics, name="audit_result").to_frame())

# 4. Missing values and conversion exceptions.
print("Missing values by original column:")
display(df_raw.isna().sum().to_frame("missing_count"))

print("Rows containing original missing values:")
display(df_raw.loc[df_raw.isna().any(axis=1)])

print("Non-missing price entries that could not be converted:")
display(
    df_raw.loc[
        df_raw["price"].notna() & df_work["price_numeric"].isna()
    ]
)

print("Non-missing date entries that could not be parsed:")
display(
    df_raw.loc[
        df_raw["date"].notna() & df_work["date_parsed"].isna()
    ]
)

# 5. Exact duplicates and repeated composite keys.
print("All occurrences of exact duplicate rows:")
display(
    df_raw.loc[df_raw.duplicated(keep=False)].sort_values(
        by=["date", "item", "city"], na_position="last"
    )
)

print("Records sharing complete date-item-city keys using original labels:")
display(
    df_raw.loc[raw_key_duplicate_mask].sort_values(
        by=raw_key_columns, na_position="last"
    )
)

# Compare keys after trimming whitespace and case-folding item labels.
# This is an audit-only comparison; original labels are not modified.
normalized_item = df_raw["item"].astype("string").str.strip().str.casefold()
normalized_keys = pd.DataFrame({
    "date": df_work["date_parsed"],
    "item_normalized": normalized_item,
    "city": df_raw["city"],
})
complete_normalized_keys = normalized_keys.notna().all(axis=1)
normalized_duplicate_mask = pd.Series(False, index=df_raw.index)
normalized_duplicate_mask.loc[complete_normalized_keys] = normalized_keys.loc[
    complete_normalized_keys,
    ["date", "item_normalized", "city"],
].duplicated(keep=False)

print("Records sharing keys after item-label normalization (audit only):")
display(
    df_raw.loc[normalized_duplicate_mask]
    .assign(item_normalized=normalized_item[normalized_duplicate_mask])
    .sort_values(
        by=["date", "city", "item_normalized"], na_position="last"
    )
)

# 6. Price validity and statistical outlier screening.
print("Rows with zero or negative converted prices:")
display(
    df_work.loc[
        df_work["price_numeric"].notna()
        & (df_work["price_numeric"] <= 0),
        df_raw.columns.tolist()
    ]
)

valid_prices = df_work["price_numeric"].dropna()
print("Descriptive statistics for valid numeric prices:")
display(valid_prices.describe().to_frame("price_statistics"))

outlier_mask = pd.Series(False, index=df_raw.index)
if not valid_prices.empty:
    q1 = valid_prices.quantile(0.25)
    q3 = valid_prices.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    print(f"IQR lower fence: {lower_fence}")
    print(f"IQR upper fence: {upper_fence}")

    outlier_mask = (
        df_work["price_numeric"].notna()
        & (
            (df_work["price_numeric"] < lower_fence)
            | (df_work["price_numeric"] > upper_fence)
        )
    )
    print("Potential IQR outliers (flagged, not removed):")
    display(
        df_raw.loc[outlier_mask]
        .assign(price_numeric=df_work.loc[outlier_mask, "price_numeric"])
    )
else:
    print("No numeric prices are available for IQR screening.")

# 7. Category, unit, source-label, and observation-frequency checks.
print("Item frequency using original labels:")
display(df_raw["item"].value_counts(dropna=False).to_frame("frequency"))

print("Item frequency after trimming whitespace and ignoring case (audit only):")
display(normalized_item.value_counts(dropna=False).to_frame("frequency"))

print("Unit label breakdown:")
display(df_raw["unit"].value_counts(dropna=False).to_frame("frequency"))

print("Source disclaimer labels:")
display(df_raw["data_label"].value_counts(dropna=False).to_frame("frequency"))

expected_per_item = (
    df_raw["city"].nunique(dropna=True)
    * df_work["date_parsed"].nunique(dropna=True)
)
print(
    "Expected observations per item if the date-city grid is balanced:",
    expected_per_item,
)
display(df_raw["item"].value_counts(dropna=False).to_frame("actual_count"))

# 8. Integrity comparison: compare only the original source columns.
# Added diagnostic columns must not inflate missing-value counts.
original_columns = df_raw.columns.tolist()
working_original_columns = df_work.loc[:, original_columns]

comparison_summary = pd.DataFrame(
    {
        "raw_state": [
            len(df_raw),
            int(df_raw.isna().sum().sum()),
            int(df_raw.duplicated().sum()),
        ],
        "working_state": [
            len(working_original_columns),
            int(working_original_columns.isna().sum().sum()),
            int(working_original_columns.duplicated().sum()),
        ],
    },
    index=[
        "total_rows",
        "original_missing_cells",
        "exact_duplicate_occurrences",
    ],
)
comparison_summary["net_difference"] = (
    comparison_summary["working_state"] - comparison_summary["raw_state"]
)
display(comparison_summary)

source_columns_unchanged = df_raw.equals(working_original_columns)
print("Original source columns unchanged:", source_columns_unchanged)
if not source_columns_unchanged:
    raise AssertionError("The original columns in the working copy have changed.")

print("Audit complete: no source rows were removed and no source prices were imputed or overwritten.")
print("Review flagged records before deciding whether any cleaning transformation is justified.")


Target file: teaching_prices_messy.csv
Dataset shape: (97, 6)
Column headers: ['date', 'item', 'city', 'price', 'unit', 'data_label']


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


,source_dtype
date,object
item,object
city,object
price,object
unit,object
data_label,object


,count,unique,top,freq
date,97,12,2026-01-08,9
item,97,5,Staple A,24
city,97,2,City 1,49
price,96,70,202.5,5
unit,97,1,PKR per kg (illustrative),97
data_label,97,1,ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,97


,audit_result
total_rows,97
total_columns,6
original_missing_cells,1
exact_duplicate_occurrences,1
raw_composite_key_duplicate_records,2
price_conversion_errors,0
non_positive_numeric_prices,0
date_conversion_errors,0
unique_dates,12
unique_cities,2


Missing values by original column:


... (27 outputs left out)

Expected observations per item if the date-city grid is balanced: 24


,actual_count
item,
Staple A,24
Oil B,24
Pulse C,24
Sweetener D,24
oil b,1


,raw_state,working_state,net_difference
total_rows,97,97,0
original_missing_cells,1,1,0
exact_duplicate_occurrences,1,1,0


Original source columns unchanged: True
Audit complete: no source rows were removed and no source prices were imputed or overwritten.
Review flagged records before deciding whether any cleaning transformation is justified.


## Reading the results

A flagged record is not automatically an incorrect record. In particular, an IQR outlier is a statistical screening result, not proof of a typing error. Any later correction should be supported by source evidence, documented, and made in a separate cleaned output.